# demo_rl19

> Phase 19 (`specs/multiagent/phase19/`): training the guard role with RL (contest credit), analysed on the Mac from the Colab outputs: H19-1 (contest credit wins back more damage than credit alone), H19-2 (varied threat makes waiting track the threat), the H19-3 constraint (guarding costs little without liars), H19-4 (arm 4's specialization: the kill criterion), H19-5 (goal drift) and H19-6 (waiting under urgency), Gate A1 and the best arm, under `RL19_PLAN`; and the power check run before `RL19_PLAN` is frozen. The speed check and the pilot's selection are read on Colab.

Artifacts (gitignored): `results/multiagent/rl/{quick_p19, eval_p19}/`. Input: `explRL_rl19.zip` unzipped into `results/multiagent/rl/colab/` (`runs_p19/`, `eval_p19/`).

- **Quick** (normal tests): synthetic evaluation rows with known effects, end to end.
- **Power** (`--flags train`, before `RL19_PLAN` is frozen): the primary tests' power from the per-run spread of Phase 17's 8 runs.
- **Main** (`--flags train`): only when the Colab output is present and `RL19_PLAN` is frozen.

In [ ]:
import json
import subprocess

from explRL.bench_grid import readCSV
from explRL.coop_grid import coopHash
from explRL.rl19_grid import RL19_PLAN
from explRL.rl19_report import figRL19, rl19Summaries, writeRL19
from explRL.rl19_synth import powerPrimary, syntheticRows19
from explRL.train import demo_artifact_dir

RL = demo_artifact_dir(None, 'multiagent/rl')
COLAB = RL / 'colab'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def analyse(mode, rows, out, nboot=2000, nNull=200, **kw):
    out.mkdir(parents=True, exist_ok=True)
    S = rl19Summaries(rows, nboot=nboot, nNull=nNull, **kw)
    meta = dict(mode=mode, planHash=coopHash(RL19_PLAN), commit=gitCommit())
    writeRL19(out / 'report.md', S, meta)
    figRL19(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, **{k: v for k, v in S.items() if k != 'perRun'}), indent=1, sort_keys=True, default=str) + '\n')
    return S

## Quick: synthetic rows

In [ ]:
qo = RL / 'quick_p19'
S = analyse('quick', syntheticRows19(), qo, nboot=200, nNull=30)
assert all((qo / f).stat().st_size > 0 for f in ('report.md', 'rl19.png', 'summary.json'))
assert S['gate'] and S['hyp']['H19_4']['supported'] and S['best'] == 'a19contest'
print(open(qo / 'report.md').read()[:1800])

## Power (`--flags train`): before `RL19_PLAN` is frozen

The two primary tests simulated with the per-run standard deviation of Phase 17's 8 `r17mix` runs (welfare gain over the warm start at 4 agents), at effects of 0.5 to 2 of those standard deviations. Record the power for the predicted effects in `RL19_PLAN['power']`; under 0.5, the README says so and states the levers (more seeds for arms 1-2, fewer for arms 3-4).

In [ ]:
#| train
import numpy as np
p17, s0 = COLAB / 'eval_p17' / 'runs.csv', COLAB / 'eval_p17s0' / 'runs.csv'
if not (p17.exists() and s0.exists()):
    print('skipped: Phase 17\'s eval_p17/runs.csv and eval_p17s0/runs.csv are not in results/multiagent/rl/colab/')
else:
    from explRL.rl17_report import RUNS, gains
    rows17 = readCSV(p17) + [r for r in readCSV(s0) if r['level'].startswith('warm|')]   # the warm start's rows are step 0's
    g = [np.mean(list(gains(rows17, r, 4).values())) for r in RUNS['r17mix']]
    sd = float(np.std(g, ddof=1))
    assert np.isfinite(sd) and sd > 0, g
    print(f"Phase 17 r17mix: per-run welfare gain {np.round(g, 4).tolist()}, standard deviation {sd:.4f}")
    for k in (0.5, 1.0, 1.5, 2.0):
        pw = powerPrimary(dict(H19_1=k * sd, H19_2=k * sd), dict(H19_1=sd, H19_2=sd), nsim=500)
        print(f'effect {k:.1f} SD: power {pw}')

## Main (`--flags train`): `RL19_PLAN`'s analysis

In [ ]:
#| train
p = COLAB / 'eval_p19' / 'runs.csv'
if not p.exists():
    print('skipped: unzip explRL_rl19.zip into results/multiagent/rl/colab/ first')
else:
    assert RL19_PLAN['frozen'], 'the analysis uses the frozen RL19_PLAN'
    runs = sorted((COLAB / 'runs_p19').glob('*_s*'))
    cut = [json.loads(x)['cut'] for d in runs for x in open(d / 'metrics.jsonl')]
    print('cut prompts in the main runs:', sum(cut), '(must be 0)')
    halts = json.loads((COLAB / 'eval_p19' / 'halts.json').read_text())
    S = analyse('full', readCSV(p), RL / 'eval_p19', halts=halts)
    print(open(RL / 'eval_p19' / 'report.md').read())